In [1]:
import os

import numpy as np
import pandas as pd

import torch
from torch import nn

/tmp/ipykernel_1011248/1099783377.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import sys, os
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.tfa import TFA

In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')
if not os.path.isdir(models_path):
    os.mkdir(models_path)

In [4]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'processed', 'ID_tf_activity.h5ad'))

In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)
sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
                                        resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
                                                'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
                                        drop_self = True, verbose = True)

tf_labels = tf_adata.var.index.unique().tolist()

ligand_labels = tf_adata.obs['sample'].unique().tolist()
ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

# filter for paths b/w ligand and tf
fn_1 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'shortest')
fn_2 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'connected')
# of the methods to identify paths, retain the one that has the most interactions
if fn_1.shape[0] > fn_2.shape[0]:
    sn_ppis = fn_1
else:
    sn_ppis = fn_2

del fn_1, fn_2

sn_ppis.loc[sn_ppis[(sn_ppis[stimulation_label] == 1) & (sn_ppis[inhibition_label] == 1)].index, 
    [stimulation_label, inhibition_label]] = [False, False]
sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label)

all_nodes = sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()
input_ligands_available = sorted(set(ligand_labels).intersection(all_nodes))

The thresholds filtered 21403  of 28277 interactions
The resources filtered 937  of 6874 interactions


100%|████████████████████████████████████| 8122/8122 [00:00<00:00, 19221.62it/s]


In [6]:
subset_tf = tf_adata[tf_adata.obs.TF_clusters.isin(['9', '15'])]
sample_size = subset_tf.obs.TF_clusters.value_counts().min()
large_cluster = subset_tf.obs.TF_clusters.value_counts().idxmax()
small_cluster = subset_tf.obs.TF_clusters.value_counts().idxmin()
large_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == large_cluster].index
small_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == small_cluster].index.tolist()
np.random.seed(seed)
lcb_sub = list(np.random.choice(large_cluster_barcodes, sample_size, replace = False))
subset_tf = subset_tf[lcb_sub + small_cluster_barcodes, :]
subset_tf.obs.TF_clusters.value_counts()

np.random.seed(seed)
selected_ligand = np.random.choice(input_ligands_available, 1)[0]

ligand_input = pd.DataFrame(subset_tf.obs.TF_clusters.map({'9': 0, '15': 1}))
ligand_input.columns = [selected_ligand]
tf_output = pd.DataFrame(subset_tf.X, index = subset_tf.obs.index, columns = subset_tf.var.index)

In [7]:
# linear scaling of inputs/outputs
projection_amplitude_in = 3
projection_amplitude_out = 1.2
# other parameters
bionet_params = {'target_steps': 100, 'max_steps': 120, 'exp_factor':50, 'tolerance': 1e-5, 'leak':1e-2} # fed directly to model

# training parameters
lr_params = {'max_iter': 5000, 
             'learning_rate': 2e-3}
other_params = {'batch_size': 256, 'noise_level': 10, 'gradient_noise_level': 1e-9}
regularization_params = {'param_lambda_L2': 1e-6, 'moa_lambda_L1': 0.1, 'ligand_lambda_L2': 1e-5, 'uniform_lambda_L2': 1e-4, 
                   'uniform_max': 1/projection_amplitude_out, 'spectral_loss_factor': 1e-5}
spectral_radius_params = {'n_probes_spectral': 5, 'power_steps_spectral': 50, 'subset_n_spectral': 10}
target_spectral_radius = 0.8
hyper_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}

# TFA
encoder_dist = None
decoder_dist = 'gauss'

e_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
d_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': torch.nn.ReLU, 'n_hidden_nodes': [64], 'linear_output': True}
ve_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}
vd_hp = {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 
         'activation_fn': nn.ReLU, 'n_hidden_nodes': [64], 'var_min': 1e-4}

Hyper_params = {
    'tfa': {'n_latent': 32, 'cat_max_norm': 1, 'encoder_dist': encoder_dist, 'decoder_dist': decoder_dist}, 
    'encoder': ve_hp if encoder_dist == 'guass' else e_hp,
    'decoder': vd_hp if decoder_dist == 'guass' else d_hp,
    'discriminator': {'batch_momentum': 0.01, 'layer_norm': False, 'dropout_rate': 0.1, 'activation_fn': nn.ReLU, 'n_hidden_nodes': [16, 16, 16], 'return_logits': True}

}

In [8]:
mod = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
                     skip_bionet_out = False,
                     covariates = subset_tf.obs,
                     categorical_covariate_keys = ['TF_clusters', 'celltype'],
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     tfa_hyper_params = Hyper_params['tfa'], 
                     encoder_hyper_params = Hyper_params['encoder'],
                     decoder_hyper_params = Hyper_params['decoder'],
                     dtype = torch.float32, device = device, seed = seed)

In [9]:
# X_in = mod.df_to_tensor(mod.X_in)[:64, :]
# X_full = mod.input_layer(X_in) # input ligands to signaling network
# Y_full = mod.signaling_network(X_full) # RNN of full signaling network
# Y_hat = mod.output_layer(Y_full) # TF outputs of signaling network

# self = mod.tf_autoencoder
# z_m, z_v, z_basal = None, None, self.encoder(Y_hat)

# Test

In [14]:
from scLEMBAS.model.scl import * 
import lightning as L

class SignalingModel(L.LightningModule):
    """Constructs the signaling network based RNN."""
    DEFAULT_TRAINING_PARAMS = {**LR_PARAMS, **OTHER_PARAMS, **REGULARIZATION_PARAMS, **SPECTRAL_RADIUS_PARAMS}
    DEFAULT_TRAINING_PARAMETERS2 = {'target_steps': 100, 'max_steps': 300, 'exp_factor': 20, 'leak': 0.01, 'tolerance': 1e-5}
    
    def __init__(self, net: pd.DataFrame, X_in: pd.DataFrame, y_out: pd.DataFrame,
                 projection_amplitude_in: Union[int, float] = 1, projection_amplitude_out: float = 1,
                 ban_list: List[str] = None, weight_label: str = 'mode_of_action', 
                 source_label: str = 'source', target_label: str = 'target', 
                bionet_params: Dict[str, float] = None , 
                 activation_function: str='MML',
                 skip_bionet_out: bool = False, 
                 covariates: Optional[pd.DataFrame] = None, categorical_covariate_keys: Optional[List[str]] = None,
                 tfa_hyper_params: Dict[str, Any] = TFA.DEFAULT_HYPER_PARAMS, 
                 encoder_hyper_params: Dict[str, Any] = Encoder.DEFAULT_HYPER_PARAMS,
                 decoder_hyper_params : Dict[str, Any] = Encoder.DEFAULT_HYPER_PARAMS,
                 
                 dtype: torch.dtype=torch.float32, device: str = 'cpu', seed: int = 888):
        """Parse the signaling network and build the model layers.

        Parameters
        ----------
        net: pd.DataFrame
            signaling network adjacency list with the following columns:
                - `weight_label`: whether the interaction is stimulating (1) or inhibiting (-1). Exclude non-interacting (0) nodes. 
                - `source_label`: source node column name
                - `target_label`: target node column name
        X_in : pd.DataFrame
            input ligand concentrations. Index represents samples and columns represent a ligand. Values represent amount of ligand introduced (e.g., concentration). 
        y_out : pd.DataFrame
            output TF activities. Index represents samples and columns represent TFs. Values represent activity of the TF. 
        ban_list : List[str], optional
            a list of signaling network nodes to disregard, by default None
        projection_amplitude_in : Union[int, float]
            value with which to scale ligand inputs by, by default 1 (see `ProjectInput` for details, can also be tuned as a learned parameter in the model)
        projection_amplitude_out : float
             value with which to scale TF activity outputs by, by default 1 (see `ProjectOutput` for details, can also be tuned as a learned parameter in the model)
        bionet_params : Dict[str, float], optional
            training parameters for the model, by default None
            Key values include:
                - 'max_steps': maximum number of time steps of the RNN, by default 300
                - 'tolerance': threshold at which to break RNN; based on magnitude of change of updated edge weight values, by default 1e-5
                - 'leak': parameter to tune extent of leaking, analogous to leaky ReLU, by default 0.01
                - 'spectral_target': _description_, by default np.exp(np.log(params['tolerance'])/params['target_steps'])
                - 'exp_factor': _description_, by default 20
        activation_function : str, optional
            RNN activation function, by default 'MML'
            options include:
                - 'MML': Michaelis-Menten-like
                - 'leaky_relu': Leaky ReLU
                - 'sigmoid': sigmoid 
        skip_bionet_out : bool, optional
            whether to skip the ProjectOutput layer when using the autoencoder, by default False.
            If True, will pass the full signaling network output directly into autoencoder
            only relevant for bulk data in the presence of covariates
        covariates : pd.DataFrame, optional
            metadata with index as sample ids and columns containing various metadata values/mappings, by default None
            If None, will run the original LEMBAS model
        categorical_covariate_keys : List[str], optional
            the columns in the dataframe representing categorical/discrete variables, by default None
        tfa_hyper_params : Dict[str, Any], optional
            Keyword arguments to pass to `TFA`. Keys include:
                n_latent: int, optional
                    dimension (no. of featuers) of the latent space, by default 64
                cat_max_norm : int | float | None, optional
                    passed to `max_norm` argument of nn.Embedding when generating categorical covariate embeddings, by default 1
                generative_decoder : bool, optional
                    whether to make the decoder layer variational/generative (True) or not (False)
                recon_loss : Literal['gauss', 'nb'], optional
                    Autoencoder loss (either "gauss" or "nb"), by default 'gauss'
                    Currently can only handle "guass"
        encoder_hyper_params : Dict[str, Any]
            Keyword arguments to pass to the `TFA` `Encoder`. Keys include:
                n_hidden_nodes : List[int], optional
                    number of hidden nodes per hidden layer, by default [64]
                    each element in the list corresponds to one hidden layer (i.e., no. of hidden layers = length of list)
                batch_momentum : float, optional
                    `momentum` parameter for `BatchNorm` layer, by default .01
                    If None, a `BatchNorm` is not added
                layer_norm : bool, optional
                    whether to have `LayerNorm` layers or not, by default False
                dropout_rate : int | float, optional
                    dropout rate to apply to each of the hidden layers, by default 0.1
                    If None, dropout is not added
                activation_fn : nn.Module | None, optional
                    non-linear Pytorch activation function, by default nn.ReLU. No activation if set to None
                linear_output : bool, optional
                    whether the final layer in the encoder should have a linear activation function (True) or the specified `activation_fn` (False)
        decoder_hyper_params : Dict[str, Any]
            same as `encoder_hyper_params`, but projects back from latent space to full feature space
            note, layer order is reversed so must list `n_hidden_nodes` as you would in encoder (from larger to bigger)
            Additional key words when using the generative/variational decoder:
                var_eps : float, optional
                    Minimum value for the variance, by default 1e-4. Used for numerical stability
        dtype : torch.dtype, optional
            datatype to store values in torch, by default torch.float32
        device : str
            whether to use gpu ("cuda") or cpu ("cpu"), by default "cpu"
        seed : int
            random seed for torch and numpy operations, by default 888
        """
        super().__init__()
        self.dtype = dtype
        self.device = device
        self.seed = seed
        self._gradient_seed_counter = 0
        self.automatic_optimization=False 
        
        self.projection_amplitude_out = projection_amplitude_out

        edge_list, node_labels, edge_MOA = self.parse_network(net, ban_list, weight_label, source_label, target_label)
        if not bionet_params:
            bionet_params = self.DEFAULT_TRAINING_PARAMETERS.copy()
        else:
            bionet_params = self.set_training_parameters(**bionet_params)

        # filter for nodes in the network, sorting by node_labels order
        self.X_in = X_in.loc[:, np.intersect1d(X_in.columns.values, node_labels)]
        self.y_out = y_out.loc[:, np.intersect1d(y_out.columns.values, node_labels)]


        # define model layers
        self.input_layer = ProjectInput(node_idx_map = self.node_idx_map, 
                                        input_labels = self.X_in.columns.values, 
                                        projection_amplitude = projection_amplitude_in, 
                                        dtype = self.dtype, 
                                        device = self.device)
        self.signaling_network = BioNet(edge_list = edge_list, 
                                        edge_MOA = edge_MOA, 
                                        n_network_nodes = len(node_labels), 
                                        bionet_params = bionet_params, 
                                        activation_function = activation_function, 
                                        dtype = self.dtype, device = self.device, seed = self.seed)

        if covariates is not None and skip_bionet_out:
            self.output_layer = None
            n_features_tfa = len(self.node_idx_map) # no. of nodes in network
        else:
            self.output_layer = ProjectOutput(node_idx_map = self.node_idx_map, 
                                              output_labels = self.y_out.columns.values, 
                                              projection_amplitude = self.projection_amplitude_out, 
                                              dtype = self.dtype, device = self.device)
            n_features_tfa = self.y_out.shape[1] # no. of TFs
        if covariates is not None:
            tfa_hyper_params = update_with_defaults(default_parameters=TFA.DEFAULT_HYPER_PARAMS, 
                                                    user_parameters = tfa_hyper_params)
            self.tf_autoencoder = TFA(covariates = covariates, 
                                      categorical_covariate_keys = categorical_covariate_keys, 
                                      n_features_in = n_features_tfa, # of TFs
                                      device = self.device, dtype = self.dtype,
                                      encoder_hyper_params = encoder_hyper_params,
                                      decoder_hyper_params = decoder_hyper_params,
                                      **tfa_hyper_params)
        else:
            self.tf_autoencoder = None

    def parse_network(self, net: pd.DataFrame, ban_list: List[str] = None, 
                 weight_label: str = 'mode_of_action', source_label: str = 'source', target_label: str = 'target'):
        """Parse adjacency network.
    
        Parameters
        ----------
        net: pd.DataFrame
            signaling network adjacency list with the following columns:
                - `weight_label`: whether the interaction is stimulating (1) or inhibiting (-1) or unknown (0). Exclude non-interacting (0)
                nodes. 
                - `source_label`: source node column name
                - `target_label`: target node column name
        ban_list : List[str], optional
            a list of signaling network nodes to disregard, by default None
    
        Returns
        -------
        edge_list : np.array
            a (2, net.shape[0]) array where the first row represents the indices for the target node and the 
            second row represents the indices for the source node. net.shape[0] is the total # of interactions
        node_labels : list
            a list of the network nodes in the same order as the indices
        edge_MOA : np.array
            a (2, net.shape[0]) array where the first row is a boolean of whether the interactions are stimulating and the 
            second row is a boolean of whether the interactions are inhibiting. 
            
            Note: Edge_list includes interactions that are not delineated as activating OR inhibiting, s.t. edge_MOA records this 
            as [False, False].
        """
        if not ban_list:
            ban_list = []
        if sorted(net[weight_label].unique()) != [-1, 0.1, 1]:
            raise ValueError(weight_label + ' values must be 1 or -1')
        
        net = net[~ net[source_label].isin(ban_list)]
        net = net[~ net[target_label].isin(ban_list)]
    
        # create an edge list with node incides
        node_labels = sorted(pd.concat([net[source_label], net[target_label]]).unique())
        self.node_idx_map = {idx: node_name for node_name, idx in enumerate(node_labels)}
        
        source_indices = net[source_label].map(self.node_idx_map).values
        target_indices = net[target_label].map(self.node_idx_map).values

        # # get edge list
        # edge_list = np.array((target_indices, source_indices))
        # edge_MOA = net[weight_label].values
        # get edge list *ordered by source-target node index*
        n_nodes = len(node_labels)
        A = scipy.sparse.csr_matrix((net[weight_label].values, (source_indices, target_indices)), shape=(n_nodes, n_nodes)) # calculate adjacency matrix
        source_indices, target_indices, edge_MOA = scipy.sparse.find(A) # re-orders adjacency list by index
        edge_list = np.array((target_indices, source_indices)) 
        edge_MOA = np.array([[edge_MOA==1],[edge_MOA==-1]]).squeeze() # convert to boolean

        return edge_list, node_labels, edge_MOA

    def df_to_tensor(self, df: pd.DataFrame):
        """Converts a pandas dataframe to the appropriate torch.tensor"""
        return torch.tensor(df.values.copy(), dtype=self.dtype, device = self.device)

    def set_training_parameters(self, **attributes):
        """Set the parameters for training the model. Overrides default parameters with attributes if specified.
        Adapted from LEMBAS `trainingParameters`
    
        Parameters
        ----------
        attributes : dict
            keys are parameter names and values are parameter value
        """
        # #set defaults
        # default_parameters = self.DEFAULT_TRAINING_PARAMETERS.copy()
        # allowed_params = list(default_parameters.keys()) + ['spectral_target']
    
        # params = {**default_parameters, **attributes}
        # if 'spectral_target' not in params.keys():
        #     params['spectral_target'] = np.exp(np.log(params['tolerance'])/params['target_steps'])
    
        # params = {k: v for k,v in params.items() if k in allowed_params}

        params = update_with_defaults(default_parameters = self.DEFAULT_TRAINING_PARAMETERS, 
                                      user_parameters = attributes, 
                                      additional_parameters = ['spectral_target'])
        if 'spectral_target' not in params.keys():
            params['spectral_target'] = np.exp(np.log(params['tolerance'])/params['target_steps'])
    
        return params

    def forward(self, X_in, categories: Optional[Dict[str, List[str]]] = None):
        """Forward pass of the model.Linearly scales ligand inputs, learns weights for signaling network interactions, 
        and transforms this to TF activity. See `forward` methods of each layer for details.

        Parameters
        ----------
        X_in : torch.tensor
            input ligand values 
        categories : Optional[Dict[str, List[str]]], optional
            a dictionary with keys as the category group and values as the 
            category label for each corresponding sample in `X_in`, by default None
            Only required if considering categorical covariates and using the autoencoder.
        """
        X_full = self.input_layer(X_in) # input ligands to signaling network
        Y_full = self.signaling_network(X_full) # RNN of full signaling network

        if self.output_layer:
            Y_hat = self.output_layer(Y_full) # TF outputs of signaling network
        else:
            Y_hat = Y_full
    
        if self.tf_autoencoder:
            if not categories:
                raise ValueError('Categorical covariates must be provided')
            z_basal, z_full, px_mean, px_var = self.tf_autoencoder(Y_hat, categories) 
        else:
            z_basal, z_full, px_mean, px_var = None, None, None, None

        return Y_hat, Y_full, z_basal, z_basal, z_full, px_mean, px_var

    def L2_reg(self, lambda_L2: Annotated[float, Ge(0)] = 0):
        """Get the L2 regularization term for the neural network parameters.
        
        Parameters
        ----------
        lambda_L2 : Annotated[float, Ge(0)]
            the regularization parameter, by default 0 (no penalty) 
        
        Returns
        -------
         : torch.Tensor
            the regularization term (as the sum of the regularization terms for each layer)
        """
        return self.input_layer.L2_reg(lambda_L2) + self.signaling_network.L2_reg(lambda_L2) + self.output_layer.L2_reg(lambda_L2)

    def ligand_regularization(self, lambda_L2: Annotated[float, Ge(0)] = 0):
        """Get the L2 regularization term for the ligand biases. Intuitively, extracellular ligands should not contribute to 
        "baseline (i.e., unstimulated) activity" affecting intrecllular signaling nodes and thus TF outputs.
        
        Parameters
        ----------
        lambda_L2 : Annotated[float, Ge(0)]
            the regularization parameter, by default 0 (no penalty) 
        
        Returns
        -------
        loss : torch.Tensor
            the regularization term
        """
        loss = lambda_L2 * torch.sum(torch.square(self.signaling_network.bias[self.input_layer.input_node_order]))
        return loss

    def uniform_regularization(self, lambda_L2: float, Y_full: torch.Tensor, 
                     target_min: float = 0.0, target_max: float = None):
        """Get the L2 regularization term for deviations of the nodes in Y_full from that of a uniform distribution between 
        `target_min` and `target_max`. 
        Note, this penalizes both deviations from the uniform distribution AND values that are out of range (like a double penalty).
    
        Parameters
        ----------
        lambda_L2 : float
            scaling factor for state loss
        Y_full : torch.Tensor
            the signaling network scaled by learned interaction weights. Shape is (samples x network nodes). 
            Output of BioNet.
        target_min : float, optional
            minimum values for nodes in Y_full to take on, by default 0.0
        target_max : float, optional
            maximum values for nodes in Y_full to take on, by default 1/`self.projection_amplitude_out`
    
        Returns
        -------
        loss : torch.Tensor
            the regularization term
        """
        lambda_L2 = torch.tensor(lambda_L2, dtype = Y_full.dtype, device = Y_full.device)
        # loss = lambda_L2 * expected_uniform_distribution(Y_full, target_max = 1/self.projectionAmplitude)
        if not target_max:
            target_max = 1/self.projection_amplitude_out
        
        sorted_Y_full, _ = torch.sort(Y_full, axis=0) # sorts each column (signaling network node) in ascending order
        target_distribution = torch.linspace(target_min, target_max, Y_full.shape[0], dtype=Y_full.dtype, device=Y_full.device).reshape(-1, 1)
        
        dist_loss = torch.sum(torch.square(sorted_Y_full - target_distribution)) # difference in distribution
        below_loss = torch.sum(Y_full.lt(target_min) * torch.square(Y_full-target_min)) # those that are below the minimum value
        above_loss = torch.sum(Y_full.gt(target_max) * torch.square(Y_full-target_max)) # those that are above the maximum value
        loss = lambda_L2*(dist_loss + below_loss + above_loss)
        return loss

    def add_gradient_noise(self, noise_level: Union[float, int]):
        """Adds noise to backwards pass gradient calculations. Use during training to make model more robust. 
    
        Parameters
        ----------
        noise_level : Union[float, int]
            scaling factor for amount of noise to add 
        """
        all_params = list(self.parameters())
        if self.seed:
            set_seeds(self.seed + self._gradient_seed_counter)
        for i in range(len(all_params)):
            if all_params[i].requires_grad:
                all_noise = torch.randn(all_params[i].grad.shape, dtype=all_params[i].dtype, device=all_params[i].device)
                all_params[i].grad += (noise_level * all_noise)
    
        self._gradient_seed_counter += 1 # new random noise each time function is called

    ########### LIGHTNING TRAINING ###########
    def configure_training(self, training_params):
        """Sets various parameters for training."""
        self.training_params = update_with_defaults(default_parameters=self.DEFAULT_TRAINING_PARAMS, 
                                                    user_parameters = training_params)
        self.loss_fn = self.training_params['loss_fn']

    def on_train_start(self):
        if not self.training_params:
            raise ValueError('Please run the "configure_training" method to specify hyper parameters for training.')

    def on_epoch_start(self):
        self.current_lr = utils.get_lr(self.current_epoch, self.training_params['max_iter'], max_height = self.training_params['learning_rate'], start_height=self.training_params['learning_rate']/10, end_height=1e-6, peak = 1000)
        self.optimizers().param_groups[0]['lr'] = self.current_lr

    def on_epoch_end(self):
        if (self.current_epoch % self.training_params['reset_epoch'] == 0) and self.current_epoch >0:
            self.optimizers().state = reset_state.copy()

    def training_step(self, batch, batch_idx):
        optimizer = self.optimizers()
        optimizer.zero_grad() # because self.automatic_optimization=False 
        X_in_, y_out_ = batch

        # forward pass
        X_full = self.input_layer(X_in_) # transform to full network with ligand input concentrations
        # utils.set_seeds(self.seed + self._gradient_seed_counter)
        network_noise = torch.randn(X_full.shape, device = X_full.device)
        X_full = X_full + (self.training_params['network_noise_scale'] * self.current_lr * network_noise) # randomly add noise to signaling network input, makes model more robust
        Y_full = self.signaling_network(X_full) # train signaling network weights
        Y_hat = self.output_layer(Y_full)

        loss = self._calculate_loss(self, y_out_, Y_hat, Y_full)
        self.log('loss', loss)

        # because self.automatic_optimization=False: 
        self.manual_backward(loss)
        self.add_gradient_noise(noise_level = self.training_params['gradient_noise_level']) # <-- reason why self.automatic_optimizer is False
        optimizer.step()
        
        return loss

    def _calculate_loss(self, y_out_, Y_hat, Y_full):
        # get prediction loss
        fit_loss = self.loss_fn(y_out_, Y_hat)
        
        # get regularization losses
        sign_reg = self.signaling_network.sign_regularization(lambda_L1 = self.training_params['moa_lambda_L1']) # incorrect MoA
        ligand_reg = self.ligand_regularization(lambda_L2 = self.training_params['ligand_lambda_L2']) # ligand biases
        stability_loss, spectral_radius = self.signaling_network.get_SS_loss(Y_full = Y_full.detach(), spectral_loss_factor = self.training_params['spectral_loss_factor'],
                                                                            subset_n = self.training_params['subset_n_spectral'], n_probes = self.training_params['n_probes_spectral'], 
                                                                            power_steps = self.training_params['power_steps_spectral'])
        uniform_reg = self.uniform_regularization(lambda_L2 = self.training_params['uniform_lambda_L2']*self.current_lr, Y_full = Y_full, 
                                                 target_min = 0, target_max = self.training_params['uniform_max']) # uniform distribution
        param_reg = self.L2_reg(self.training_params['param_lambda_L2']) # all model weights and signaling network biases
        
        total_loss = fit_loss + sign_reg + ligand_reg + param_reg + stability_loss + uniform_reg
        self.log('fit_loss', fit_loss); self.log('spectral_radius', spectral_radius)
    
    def copy(self):
        return copy.deepcopy(self)



In [ ]:
BASE_TRAINING_PARAMS = {'optimizer': torch.optim.Adam, 'loss_fn': torch.nn.MSELoss(reduction='mean')}
LR_PARAMS = {'max_epoch': 5000, 'learning_rate': 2e-3}
OTHER_PARAMS = {'batch_size': 32, 'network_noise_scale': 10, 'gradient_noise_level': 1e-9}
REGULARIZATION_PARAMS = {'param_lambda_L2': 1e-6, 'moa_lambda_L1': 0.1, 'ligand_lambda_L2': 1e-5, 'uniform_lambda_L2': 1e-4, 
                   'uniform_max': (1/1.2), 'spectral_loss_factor': 1e-5}
SPECTRAL_RADIUS_PARAMS = {'n_probes_spectral': 5, 'power_steps_spectral': 50, 'subset_n_spectral': 10}
DEFAULT_TRAINING_PARAMS = {**BASE_TRAINING_PARAMS, **LR_PARAMS, **OTHER_PARAMS, **REGULARIZATION_PARAMS, **SPECTRAL_RADIUS_PARAMS}